# WarBias pilot, blinded selection and full generation

The paper workflow compares several candidate models using two prompting strategies on a subset, obtains blinded human assessments, records the selected condition, then generates the full source-language matrix. Translation follows the completed full run in the companion notebook.

The original retained comparison used Ukrainian, two models (Lapa 12B and Mamay 27B), and zero-shot versus sequential S→C→U prompting. Human assessment was of the pilot; it did not validate every full-run item. Read the [construction protocol](../../docs/warbias-construction.md) for verified historical counts, prompt/selection provenance gaps and the separate later English-first lineage.

This executable notebook uses two synthetic candidate models, fixed local responses and clearly labelled score/choice fixtures. It exercises the same pipeline functions, without performing or claiming actual human assessment, new model generation or historical replay.

The stages live in small modules: `pilot.freeze_pilot` / `pilot.run_pilot`; `blind_review.build_blind_package` / `bind_reviews`; `selection.select_condition`; `production.generate_full`. Public reviewer forms exclude model/strategy identities; the identity key and post-selection receipt remain private. Sequential prompting carries exact saved S into C and exact S+C into U, with no inserted human review between calls.

Install the paired Fair-UK checkout and this repository before running. No network request or checkpoint download occurs.


In [ ]:
import tempfile
from pathlib import Path

from fairforget.examples import run_generation_example

work = Path(tempfile.mkdtemp(prefix="fairforget-software-check-"))
prompt_root = next(
    candidate / "prompts"
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base / "warbias", base)
    if (candidate / "prompts/manifest.json").is_file()
)

In [ ]:
result = run_generation_example(work / "run", prompt_root=prompt_root)
assert result["qualification"] == "synthetic_software_check"
result

In [ ]:
import json

# Inspect the portable evidence after the synthetic lifecycle.
plan = json.loads((work / "run/pilot/pilot-plan.json").read_text())
public_panel = json.loads((work / "run/review/public-package.json").read_text())
full_run = json.loads((work / "run/full/production-result.json").read_text())
print(
    {
        "candidate_models": len(plan["candidates"]),
        "prompting_strategies": list(plan["strategies"]),
        "blinded_examples": len(public_panel["rows"]),
        "full_run_status": full_run["status"],
        "actual_human_validation": result["human_validated"],
    }
)